# Optimisation from scratch: gradient descent and its pitfalls

*Mathematics for Machine Learning from First Principles · Lesson 5*

Training a model means searching for the parameters that make a loss as small as possible. In this notebook you will build that search, gradient descent, in plain Python. You will see how the learning rate decides between slow progress, fast convergence and divergence; decide when to stop; meet local minima and numerical overflow; and finally fit a straight line to data and confirm the answer with the exact least-squares formulas.

## Learning objectives

By the end of this notebook you will be able to:

1. express a learning problem as the minimisation of an objective (loss) function;
2. implement gradient descent for functions of one and of several variables;
3. recognise a learning rate that is too small, suitable or too large, and detect divergence with `math.isfinite`;
4. stop an optimiser with a gradient-norm tolerance, a loss-change tolerance and an iteration cap;
5. explain why gradient descent can reach different minima from different starting points, and what convexity guarantees;
6. recognise overflow and compute $\log(1 + e^{x})$ in a numerically stable way;
7. fit a line to data by gradient descent on the mean squared error, and check it against the closed-form least-squares solution.

## Dependencies

**Tier 0 — Python standard library only.** The core cells import only `math`, `random` and `statistics`, which come with every Python installation. The notebook runs in JupyterLite in your browser and on the course JupyterHub server without installing anything.

Five cells are optional. Four are tagged `visualization` and draw plots with Matplotlib (and NumPy); one is tagged `comparison` and checks the line fit against NumPy. Both libraries are available in JupyterLite and on the course server. Nothing later in the notebook depends on these cells, and the core cells print tables with the same information, so you can skip them.

**How to use this notebook.** Run the cells in order from the top; *Run → Run All Cells* also works. The exercises near the end start as stubs that raise `NotImplementedError`, so the whole notebook runs cleanly before you attempt them. Each exercise is followed by a `check(...)` cell, which prints `not implemented yet` until you write your answer, confirms a correct answer, and stops with an `AssertionError` that names the failing case when an answer is wrong. Reference solutions are in the final section; attempt each exercise before you read them.

In [ ]:
import math
import random
import statistics


def _matches(actual, expected, rel_tol, abs_tol):
    """Compare a result with an expected value; numbers are compared approximately."""
    if expected is None:
        return actual is None
    if isinstance(expected, bool):
        return isinstance(actual, bool) and actual == expected
    if isinstance(expected, (int, float)):
        return (
            isinstance(actual, (int, float))
            and not isinstance(actual, bool)
            and math.isclose(actual, expected, rel_tol=rel_tol, abs_tol=abs_tol)
        )
    if isinstance(expected, (list, tuple)):
        return (
            isinstance(actual, (list, tuple))
            and len(actual) == len(expected)
            and all(_matches(a, e, rel_tol, abs_tol) for a, e in zip(actual, expected))
        )
    return actual == expected


def check(name, fn, cases, rel_tol=1e-6, abs_tol=1e-9):
    """Test an exercise function and report the outcome.

    Each case is a tuple (description, args, expected), and the function is
    called as fn(*args). `expected` is either a value (numbers are compared
    with math.isclose, lists element by element) or a function that receives
    the result and returns True when the result is acceptable.

    Prints "<name>: not implemented yet" and returns False while fn raises
    NotImplementedError. Prints a pass line and returns True when every case
    passes. Raises AssertionError, naming the failing case, when a result is wrong.
    """
    for description, args, expected in cases:
        try:
            result = fn(*args)
        except NotImplementedError:
            print(f"{name}: not implemented yet")
            return False
        if callable(expected):
            try:
                passed = bool(expected(result))
            except Exception:  # the result does not have the expected shape
                passed = False
            wanted = "a result that matches the description"
        else:
            passed = _matches(result, expected, rel_tol, abs_tol)
            wanted = repr(expected)
        if not passed:
            hint = " Did you forget a return statement?" if result is None else ""
            raise AssertionError(f"{name}: {description}: got {result!r}, expected {wanted}.{hint}")
    count = len(cases)
    print(f"{name}: correct ({count} check{'' if count == 1 else 's'} passed)")
    return True


print("The check() helper is ready.")

## Theory

### Objective functions

An **objective function** scores a choice of parameters $\theta$; in machine learning it is usually a **loss** $L(\theta)$, where smaller is better. Optimisation looks for a **minimiser**

$$\theta^\star = \arg\min_{\theta} L(\theta) .$$

For a model that predicts $\hat y_i$ for targets $y_i$, a standard loss is the **mean squared error**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\hat y_i - y_i)^2 .$$

### Gradient descent

The gradient $\nabla L(\theta)$ points in the direction of steepest increase, so a small step against it lowers the loss. Gradient descent repeats the update

$$\theta_{t+1} = \theta_t - \eta\,\nabla L(\theta_t),$$

where the **learning rate** $\eta > 0$ sets the size of each step. A first-order Taylor expansion shows why this works when $\eta$ is small enough:

$$L\bigl(\theta_t - \eta\,\nabla L(\theta_t)\bigr) \approx L(\theta_t) - \eta\,\lVert \nabla L(\theta_t) \rVert^2 \le L(\theta_t) .$$

### How the learning rate decides the outcome

For the one-dimensional quadratic $L(x) = (x - 3)^2 + 1$, the gradient is $2(x - 3)$, and one update gives

$$x_{t+1} - 3 = (1 - 2\eta)\,(x_t - 3) .$$

The distance to the minimum is multiplied by $1 - 2\eta$ at every step:

- for $0 < \eta < 0.5$ the factor lies between $0$ and $1$, and the iterates move steadily towards $3$; the smaller $\eta$, the slower;
- $\eta = 0.5$ lands on the minimum in a single step;
- for $0.5 < \eta < 1$ each step overshoots, but the distance still shrinks, so the iterates converge while oscillating;
- $\eta = 1$ bounces between two points for ever;
- for $\eta > 1$ the distance grows at every step: the run **diverges**.

In general, curvature limits the learning rate: along a direction in which the second derivative of the loss is $a$, gradient descent is stable only if $\eta < 2/a$.

### When to stop

Gradient descent rarely lands exactly on a minimum, so it needs a rule for stopping. Common criteria are:

- the gradient is small, $\lVert \nabla L(\theta_t) \rVert \le \varepsilon_{\text{grad}}$, because the gradient vanishes at a minimum;
- the loss has stopped changing, $|L(\theta_{t-1}) - L(\theta_t)| \le \varepsilon_{\text{loss}}$;
- an **iteration cap**, $t \ge T_{\max}$, a safety net that guarantees the loop ends;
- divergence: the loss is no longer a finite number.

### Convexity

A function is **convex** if every chord lies on or above its graph:

$$L\bigl(\lambda a + (1 - \lambda) b\bigr) \le \lambda L(a) + (1 - \lambda) L(b) \quad \text{for all } a, b \text{ and all } 0 \le \lambda \le 1 .$$

For a twice-differentiable function of one variable, this is the same as $L''(x) \ge 0$ everywhere. On a convex function every local minimum is a global minimum, so gradient descent with a suitable learning rate cannot be trapped in the wrong valley. The mean squared error of a linear model is convex in its parameters; the loss of a neural network is not.

## Step-by-step implementation

Each step implements one idea in a code cell, tests it with `assert` statements in the next cell, and then states the behaviour you should see. If an assertion fails, its message names the quantity that went wrong.

### Step 1 — Objective functions and their gradients

We define the objectives used throughout the notebook, each with its gradient:

- `quadratic(x)` $= (x - 3)^2 + 1$, a one-dimensional bowl whose minimum value $1$ is at $x = 3$;
- `bowl(theta)` $= (x - 1)^2 + 10\,(y + 2)^2$, a two-dimensional bowl, ten times steeper in $y$ than in $x$, whose minimum value $0$ is at $(1, -2)$;
- `mean_squared_error(predictions, targets)`, the loss we minimise in Step 9.

We also bring back the central-difference gradient from Lesson 4 to check every gradient we write. Notice that squares are written as `d * d` rather than `d ** 2`. Both give the same answer for ordinary numbers, but when a result is too large to represent, `*` returns `inf` while `**` raises `OverflowError`. Step 4 relies on seeing `inf`.

In [ ]:
def quadratic(x):
    d = x - 3.0
    return d * d + 1.0


def quadratic_grad(x):
    return 2.0 * (x - 3.0)


def bowl(theta):
    x, y = theta
    dx, dy = x - 1.0, y + 2.0
    return dx * dx + 10.0 * dy * dy


def bowl_grad(theta):
    x, y = theta
    return [2.0 * (x - 1.0), 20.0 * (y + 2.0)]


def mean_squared_error(predictions, targets):
    """Average squared difference between predictions and targets."""
    if len(predictions) == 0:
        raise ValueError("mean_squared_error needs at least one prediction")
    total = 0.0
    for prediction, target in zip(predictions, targets, strict=True):
        error = prediction - target
        total += error * error
    return total / len(predictions)


def numerical_gradient(f, theta, h=1e-5):
    """Central-difference gradient, as in Lesson 4."""
    gradient = []
    for i in range(len(theta)):
        ahead, behind = list(theta), list(theta)
        ahead[i] += h
        behind[i] -= h
        gradient.append((f(ahead) - f(behind)) / (2 * h))
    return gradient


print(f"quadratic(3) = {quadratic(3.0)}, quadratic(0) = {quadratic(0.0)}")
print(f"bowl([1, -2]) = {bowl([1.0, -2.0])}, bowl([0, 0]) = {bowl([0.0, 0.0])}, gradient there: {bowl_grad([0.0, 0.0])}")
print(f"MSE of predictions [2, 4, 7] for targets [1, 4, 9]: {mean_squared_error([2, 4, 7], [1, 4, 9]):.4f}")

In [ ]:
# Tests for Step 1
assert quadratic(3.0) == 1.0 and quadratic(0.0) == 10.0, "quadratic should be (x - 3)**2 + 1"
assert quadratic_grad(3.0) == 0.0, "the gradient vanishes at the minimum"
assert bowl([1.0, -2.0]) == 0.0 and bowl([0.0, 0.0]) == 41.0, "bowl should be (x - 1)**2 + 10 * (y + 2)**2"
assert bowl_grad([1.0, -2.0]) == [0.0, 0.0], "the gradient vanishes at the minimum"
assert math.isclose(mean_squared_error([2, 4, 7], [1, 4, 9]), 5 / 3), "(1 + 0 + 4) / 3 = 5/3"
assert mean_squared_error([1.5, 2.5], [1.5, 2.5]) == 0.0, "perfect predictions have zero loss"
try:
    mean_squared_error([1.0, 2.0], [1.0])
except ValueError:
    pass
else:
    raise AssertionError("predictions and targets of different lengths should raise ValueError")
for theta in ([0.0, 0.0], [2.5, -1.0], [-1.0, 3.0]):
    for analytic, numerical in zip(bowl_grad(theta), numerical_gradient(bowl, theta)):
        assert math.isclose(analytic, numerical, rel_tol=1e-7, abs_tol=1e-7), f"bowl_grad disagrees with a central difference at {theta}"
    assert math.isclose(quadratic_grad(theta[0]), numerical_gradient(lambda t: quadratic(t[0]), [theta[0]])[0], rel_tol=1e-7, abs_tol=1e-7)
print("Step 1 tests passed.")

**Expected behaviour.** The cell prints `quadratic(3) = 1.0` and `quadratic(0) = 10.0`; `bowl([1, -2]) = 0.0` and `bowl([0, 0]) = 41.0`, where the gradient is `[-2.0, 40.0]`, twenty times larger in $y$ than in $x$; and a mean squared error of $1.6667$. Both analytic gradients agree with central differences. The test cell prints `Step 1 tests passed.`

### Step 2 — Gradient descent in one dimension

In one dimension the update rule reads $x_{t+1} = x_t - \eta\,L'(x_t)$. `gradient_descent_1d` applies it a fixed number of times and returns every iterate, so that we can inspect the whole path. From $x_0 = 0$ with $\eta = 0.1$, the first step is $x_1 = 0 - 0.1 \times 2\,(0 - 3) = 0.6$, and each step removes 20% of the remaining distance to $3$.

In [ ]:
def gradient_descent_1d(grad, x0, lr, steps):
    """Apply x <- x - lr * grad(x) `steps` times; return all iterates, starting with x0."""
    xs = [x0]
    x = x0
    for _ in range(steps):
        x = x - lr * grad(x)
        xs.append(x)
    return xs


path = gradient_descent_1d(quadratic_grad, x0=0.0, lr=0.1, steps=60)
print(f"{'t':>3}  {'x_t':>12}  {'L(x_t)':>14}  {'gradient':>10}")
for t in (0, 1, 2, 3, 5, 10, 20, 40, 60):
    x = path[t]
    print(f"{t:>3}  {x:>12.8f}  {quadratic(x):>14.10f}  {quadratic_grad(x):>10.2e}")

In [ ]:
# Tests for Step 2
assert len(path) == 61 and path[0] == 0.0, "the path should hold x0 and one value per step"
assert math.isclose(path[1], 0.6), "the first step from 0 with lr = 0.1 should reach 0.6"
assert math.isclose(path[2], 1.08), "the second step should reach 0.6 + 0.1 * 4.8 = 1.08"
losses_1d = [quadratic(x) for x in path]
assert all(later <= earlier for earlier, later in zip(losses_1d, losses_1d[1:])), "with this learning rate the loss must never increase"
assert math.isclose(path[-1], 3.0, abs_tol=1e-5), "after 60 steps x should be within 1e-5 of the minimum at 3"
print("Step 2 tests passed.")

**Expected behaviour.** The distance to $3$ shrinks by a factor of $0.8$ per step: $x_t$ runs $0, 0.6, 1.08, 1.464, \ldots$, and the gradient shrinks by the same factor. After 60 steps $x \approx 2.9999954$, and the loss is within about $2 \times 10^{-11}$ of its minimum value $1$. The test cell prints `Step 2 tests passed.`

### Step 3 — Gradient descent in several dimensions

With several parameters, $\theta$ is a list, and the update subtracts $\eta$ times each component of the gradient from the matching parameter. `gradient_descent` returns the parameters and the loss at every step. It also guards against divergence: if the loss is no longer a finite number, the run has blown up and further steps could only produce `inf` or `nan`, so the loop stops. Step 4 puts that guard to work.

We run it on the bowl with $\eta = 0.09$. Along $y$ the curvature is $20$, so the stability limit is $2/20 = 0.1$: the learning rate is close to it, and $y$ overshoots $-2$ at every step, while $x$, with curvature $2$, creeps towards $1$. The result is the zig-zag typical of valleys that are steeper in one direction than another.

To use the same function on the one-dimensional quadratic, we also wrap `quadratic` so that it takes and returns lists holding a single number.

In [ ]:
def gradient_descent(f, grad, theta0, lr, steps):
    """Gradient descent on a list of parameters.

    Applies theta <- theta - lr * grad(theta) up to `steps` times and returns
    (thetas, losses): every iterate, starting with theta0, and its loss.
    Stops early when the loss is no longer finite, because the run has diverged.
    """
    theta = list(theta0)
    thetas, losses = [theta], [f(theta)]
    for _ in range(steps):
        gradient = grad(theta)
        theta = [t - lr * g for t, g in zip(theta, gradient, strict=True)]  # a new list each step
        try:
            loss = f(theta)
        except OverflowError:  # raised by ** and math.exp when a result is too large
            loss = math.inf
        thetas.append(theta)
        losses.append(loss)
        if not math.isfinite(loss):
            break
    return thetas, losses


def quadratic_vec(theta):
    """quadratic() for a list holding one number."""
    return quadratic(theta[0])


def quadratic_vec_grad(theta):
    return [quadratic_grad(theta[0])]


bowl_thetas, bowl_losses = gradient_descent(bowl, bowl_grad, theta0=[-2.0, 1.0], lr=0.09, steps=200)
print(f"{'step':>4}  {'x':>10}  {'y':>10}  {'loss':>10}")
for t in (0, 1, 2, 3, 4, 5, 10, 25, 50, 100, 200):
    x, y = bowl_thetas[t]
    print(f"{t:>4}  {x:>10.6f}  {y:>10.6f}  {bowl_losses[t]:>10.3e}")

In [ ]:
# Tests for Step 3
assert bowl_thetas[0] == [-2.0, 1.0] and len(bowl_thetas) == len(bowl_losses) == 201
assert all(math.isclose(a, b, abs_tol=1e-9) for a, b in zip(bowl_thetas[-1], [1.0, -2.0])), "the run should converge to (1, -2)"
assert all(later <= earlier for earlier, later in zip(bowl_losses, bowl_losses[1:])), "the loss should decrease at every step"
y_offsets = [theta[1] + 2.0 for theta in bowl_thetas[:6]]
assert all(a * b < 0 for a, b in zip(y_offsets, y_offsets[1:])), "y should overshoot -2 and change side at every step"
# In one dimension, gradient_descent reproduces gradient_descent_1d from Step 2.
one_d_thetas, _ = gradient_descent(quadratic_vec, quadratic_vec_grad, [0.0], 0.1, 60)
assert math.isclose(one_d_thetas[-1][0], path[-1]), "the vector version should match Step 2"
start = [-2.0, 1.0]
gradient_descent(bowl, bowl_grad, start, 0.09, 5)
assert start == [-2.0, 1.0], "gradient_descent must not modify theta0"
print("Step 3 tests passed.")

**Expected behaviour.** In the table, $y$ jumps from $1$ to $-4.4$, then to $-0.08$, then to $-3.536$: it changes sides of $-2$ at every step, while its distance from $-2$ shrinks by a factor of $0.8$. Meanwhile $x$ moves steadily from $-2$ towards $1$, closing 18% of the remaining gap per step. The loss falls at every step; after 200 steps it is about $2 \times 10^{-30}$, and the parameters print as exactly $(1, -2)$. The test cell prints `Step 3 tests passed.`

**Optional visualisation.** The next cell is tagged `visualization` and draws the first 30 steps of the path over contour lines of the bowl, using NumPy and Matplotlib. It is optional: the table above shows the same path, and nothing later uses this cell.

In [ ]:
# Optional: needs NumPy and Matplotlib, which JupyterLite and the course server provide.
try:
    import numpy as np
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("NumPy or Matplotlib is not available here; the table above shows the same path.")

if plt is not None:
    grid_x, grid_y = np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-4.8, 1.5, 200))
    heights = (grid_x - 1) ** 2 + 10 * (grid_y + 2) ** 2
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.contour(grid_x, grid_y, heights, levels=[0.1, 0.5, 1, 2, 5, 10, 20, 40, 80], cmap="viridis")
    ax.plot([t[0] for t in bowl_thetas[:30]], [t[1] for t in bowl_thetas[:30]], marker="o", markersize=3, label="gradient descent, lr = 0.09")
    ax.plot([1], [-2], marker="*", markersize=14, linestyle="none", label="minimum (1, -2)")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Gradient descent zig-zags across the steep direction")
    ax.legend(loc="upper left")
    plt.show()

### Step 4 — Too small, just right, too large

We run gradient descent on the quadratic from $x_0 = 0$, for at most 1,000 steps, with five learning rates. The theory section predicts the outcome from the factor $1 - 2\eta$: it is $0.998$ for $\eta = 0.001$ (painfully slow), $0.8$ for $\eta = 0.1$ (fast), $-0.8$ for $\eta = 0.9$ (oscillating but converging), $-1$ for $\eta = 1$ (bouncing between $0$ and $6$ for ever) and $-2$ for $\eta = 1.5$ (diverging). A diverging run doubles its distance from the minimum at every step, so its loss overflows to `inf` after roughly 500 steps; `math.isfinite` detects this, and `gradient_descent` stops.

`describe_run` turns a list of losses into a verdict, and the second table shows the loss after selected numbers of steps.

In [ ]:
def describe_run(losses, minimum=1.0, tol=1e-9):
    """Summarise a run from its losses; `minimum` is the best achievable loss."""
    if not math.isfinite(losses[-1]):
        return f"diverged: the loss became {losses[-1]} at step {len(losses) - 1}"
    if losses[-1] - minimum <= tol:
        first = next(t for t, loss in enumerate(losses) if loss - minimum <= tol)
        return f"converged: within {tol:g} of the minimum after {first} steps"
    return f"not converged: the loss is still {losses[-1]:.6g} after {len(losses) - 1} steps"


LEARNING_RATES = [0.001, 0.1, 0.9, 1.0, 1.5]
runs = {}
for lr in LEARNING_RATES:
    _, runs[lr] = gradient_descent(quadratic_vec, quadratic_vec_grad, [0.0], lr, 1000)
    print(f"lr = {lr:<5}  {describe_run(runs[lr])}")

print("\nLoss after t steps:")
print(f"{'t':>4}" + "".join(f"{'lr = ' + str(lr):>13}" for lr in LEARNING_RATES))
for t in (0, 1, 2, 3, 4, 5, 10, 50, 100):
    print(f"{t:>4}" + "".join(f"{runs[lr][t]:>13.6g}" for lr in LEARNING_RATES))

In [ ]:
# Tests for Step 4
assert runs[0.001][-1] > 1.1, "lr = 0.001 should still be far from the minimum after 1000 steps"
assert math.isclose(runs[0.1][-1], 1.0, abs_tol=1e-12), "lr = 0.1 should converge"
assert math.isclose(runs[0.9][-1], 1.0, abs_tol=1e-12), "lr = 0.9 should converge too, while oscillating"
assert all(loss == 10.0 for loss in runs[1.0]), "lr = 1 should bounce between 0 and 6, where the loss is always 10"
assert not math.isfinite(runs[1.5][-1]), "lr = 1.5 should diverge until the loss overflows"
assert len(runs[1.5]) < 1001, "gradient_descent should stop as soon as the loss is no longer finite"
print("Step 4 tests passed.")

**Expected behaviour.** The verdicts read: $\eta = 0.001$ has not converged (the loss is still about $1.164$ after 1,000 steps); $\eta = 0.1$ and $\eta = 0.9$ both converge to within $10^{-9}$ of the minimum after 52 steps, because $|1 - 2\eta| = 0.8$ for both; $\eta = 1$ never converges, because its loss stays at exactly $10$; and $\eta = 1.5$ diverges, its loss becoming `inf` at step 511. In the second table the loss for $\eta = 1.5$ grows roughly fourfold per step and has passed $10^{60}$ by step 100. The fastest rate for this function would be $\eta = 0.5$, but that is a property of its curvature: the bowl of Step 3 diverges for any $\eta > 0.1$. The test cell prints `Step 4 tests passed.`

**Optional visualisation.** The next cell is tagged `visualization` and plots the gap between each run's loss and the minimum value $1$ on a logarithmic axis, for the first 40 steps. It is optional: the second table above shows the same numbers, and nothing later uses this cell. The lines for $\eta = 0.1$ and $\eta = 0.9$ lie on top of each other, because both shrink the distance by the same factor.

In [ ]:
# Optional: needs Matplotlib, which JupyterLite and the course server provide.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Matplotlib is not available here; the table above shows the same results.")

if plt is not None:
    fig, ax = plt.subplots(figsize=(6.5, 4))
    for lr, marker in zip(LEARNING_RATES, ["o", "s", "x", "^", "v"]):
        gaps = [loss - 1.0 for loss in runs[lr][:41]]
        ax.semilogy(range(len(gaps)), gaps, marker=marker, markersize=3, label=f"lr = {lr}")
    ax.set_xlabel("step t")
    ax.set_ylabel("L(x_t) - 1 (log scale)")
    ax.set_title("The learning rate decides between crawling, converging and diverging")
    ax.legend()
    plt.show()

### Step 5 — Convergence and stopping criteria

A fixed number of steps either wastes time after convergence or stops too early. `stopping_reason` turns the criteria from the theory section into code. Given the number of updates made so far, the previous and current losses, and the current gradient, it returns the reason to stop, or `None` to carry on. It checks divergence first, then the gradient norm, then the change in loss, and finally the iteration cap.

To see the criteria at work, we record a long run on the bowl with $\eta = 0.01$ and ask, for each criterion on its own, at which step it would first have stopped the run, and how far from the minimum the parameters were at that point.

In [ ]:
DIVERGED = "diverged"
GRADIENT_SMALL = "gradient norm below tolerance"
LOSS_STALLED = "loss change below tolerance"
CAP_REACHED = "iteration cap reached"


def vector_norm(v):
    """Euclidean length of a list of numbers."""
    return math.sqrt(sum(component * component for component in v))


def stopping_reason(step, previous_loss, loss, gradient, grad_tol=1e-6, loss_tol=1e-14, max_steps=10_000):
    """Return the reason to stop gradient descent now, or None to continue.

    `step` is the number of updates made so far; `previous_loss` is None before the first update.
    """
    if not math.isfinite(loss):
        return DIVERGED
    if vector_norm(gradient) <= grad_tol:
        return GRADIENT_SMALL
    if previous_loss is not None and abs(previous_loss - loss) <= loss_tol:
        return LOSS_STALLED
    if step >= max_steps:
        return CAP_REACHED
    return None


long_thetas, long_losses = gradient_descent(bowl, bowl_grad, [-2.0, 1.0], lr=0.01, steps=3000)
criteria = {
    "gradient norm <= 1e-6": lambda t: vector_norm(bowl_grad(long_thetas[t])) <= 1e-6,
    "loss change <= 1e-14": lambda t: t > 0 and abs(long_losses[t - 1] - long_losses[t]) <= 1e-14,
    "loss change <= 1e-6": lambda t: t > 0 and abs(long_losses[t - 1] - long_losses[t]) <= 1e-6,
    "cap of 500 steps": lambda t: t >= 500,
}
stop_steps = {}
print(f"{'criterion':<22}  {'first stops at step':>19}  {'distance to the minimum':>23}")
for name, fires in criteria.items():
    t = next(t for t in range(len(long_thetas)) if fires(t))
    stop_steps[name] = t
    distance = vector_norm([long_thetas[t][0] - 1.0, long_thetas[t][1] + 2.0])
    print(f"{name:<22}  {t:>19}  {distance:>23.1e}")

# "False convergence": one update with a tiny learning rate barely changes the loss.
tiny_step = [t - 1e-13 * g for t, g in zip([-2.0, 1.0], bowl_grad([-2.0, 1.0]))]
print(f"\nAfter one update with lr = 1e-13: loss change {bowl([-2.0, 1.0]) - bowl(tiny_step):.1e}, "
      f"gradient norm {vector_norm(bowl_grad(tiny_step)):.1f}")

In [ ]:
# Tests for Step 5
assert stopping_reason(3, 2.0, math.inf, [1.0]) == DIVERGED
assert stopping_reason(3, 2.0, math.nan, [1.0]) == DIVERGED
assert stopping_reason(3, 2.0, 1.0, [1e-7, -1e-7]) == GRADIENT_SMALL
assert stopping_reason(3, 1.0, 1.0 - 1e-15, [0.5]) == LOSS_STALLED
assert stopping_reason(0, None, 5.0, [0.5]) is None, "before the first update there is no loss change to measure"
assert stopping_reason(10_000, 2.0, 1.0, [0.5]) == CAP_REACHED
assert stopping_reason(10, 2.0, 1.0, [0.5]) is None
assert stop_steps["loss change <= 1e-6"] < stop_steps["cap of 500 steps"] < stop_steps["gradient norm <= 1e-6"]
# A tiny learning rate makes the change in loss tiny far from the minimum: "false convergence".
assert stopping_reason(1, bowl([-2.0, 1.0]), bowl(tiny_step), bowl_grad(tiny_step), loss_tol=1e-9) == LOSS_STALLED
assert vector_norm(bowl_grad(tiny_step)) > 50, "... even though the gradient is still large"
print("Step 5 tests passed.")

**Expected behaviour.** The loose loss-change tolerance of $10^{-6}$ fires first, at step 318, while the parameters are still about $5 \times 10^{-3}$ from the minimum. The cap of 500 steps stops at about $1 \times 10^{-4}$. The gradient-norm tolerance of $10^{-6}$ and the strict loss-change tolerance of $10^{-14}$ both wait until around step 770, by which time the parameters are within about $5 \times 10^{-7}$ of the minimum. The criteria measure different things, so a tolerance only means something together with the quantity it limits. The last line shows the trap: after one update with $\eta = 10^{-13}$ the loss has changed by only about $3.6 \times 10^{-10}$, so a loss-change rule would declare convergence, yet the gradient norm is still about $60$. The test cell prints `Step 5 tests passed.`

### Step 6 — Local and global minima

Real losses are rarely simple bowls. The function

$$w(x) = x^4 - 3x^2 + x, \qquad w'(x) = 4x^3 - 6x + 1,$$

has two valleys: a deep one near $x \approx -1.30$, the **global minimum**, and a shallower one near $x \approx 1.13$, a **local minimum**. A hump near $x \approx 0.17$ separates them. Gradient descent only ever walks downhill, so the valley it ends in depends on where it starts. We run it from several starting points, then try ten random starts from a seeded generator and keep the best result, a simple strategy called **random restarts**.

In [ ]:
def double_well(x):
    x2 = x * x
    return x2 * x2 - 3.0 * x2 + x


def double_well_grad(x):
    return 4.0 * x * x * x - 6.0 * x + 1.0


def double_well_curvature(x):
    """The second derivative, 12x**2 - 6."""
    return 12.0 * x * x - 6.0


print(f"{'start':>6}  {'end':>10}  {'loss at end':>11}  {'gradient':>9}  {'curvature':>9}")
endpoints = {}
for start in (-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0):
    end = gradient_descent_1d(double_well_grad, start, lr=0.01, steps=1000)[-1]
    endpoints[start] = end
    print(f"{start:>6.1f}  {end:>10.6f}  {double_well(end):>11.6f}  {double_well_grad(end):>9.1e}  {double_well_curvature(end):>9.2f}")

rng = random.Random(42)  # a fixed seed, so every run uses the same starting points
starts = [rng.uniform(-2.0, 2.0) for _ in range(10)]
results = [gradient_descent_1d(double_well_grad, s, lr=0.01, steps=1000)[-1] for s in starts]
best = min(results, key=double_well)
print(f"\nRandom restarts from {len(starts)} seeded starting points:")
print(f"  {sum(r < 0 for r in results)} ended in the left valley and {sum(r > 0 for r in results)} in the right valley")
print(f"  best end point: x = {best:.6f}, loss {double_well(best):.6f}")

In [ ]:
# Tests for Step 6
left, right = endpoints[-2.0], endpoints[2.0]
assert left < 0 < right, "starts at -2 and at 2 should end in different valleys"
for x in (left, right):
    assert abs(double_well_grad(x)) < 1e-8, f"the end point {x} should be a stationary point"
    assert double_well_curvature(x) > 0, "... with positive curvature, so a minimum"
assert double_well(left) < double_well(right), "the left valley is the global minimum"
assert endpoints[0.0] < 0 < endpoints[0.5], "the hump between x = 0 and x = 0.5 separates the two valleys"
assert math.isclose(best, left, abs_tol=1e-6), "random restarts should find the global minimum"
print("Step 6 tests passed.")

**Expected behaviour.** Starts at $-2$, $-1$, $-0.5$ and $0$ end at $x \approx -1.300840$ with loss $\approx -3.514$; starts at $0.5$, $1$ and $2$ end at $x \approx 1.130901$ with loss $\approx -1.070$. Every end point has a gradient of essentially zero and positive curvature, so each is a genuine minimum, but only the left one is global. The start at $0$ goes left because $w'(0) = 1 > 0$, while the start at $0.5$ goes right because $w'(0.5) = -1.5 < 0$. Of the ten random restarts, six end in the left valley and four in the right one, and the best of them is the global minimum. The test cell prints `Step 6 tests passed.`

**Optional visualisation.** The next cell is tagged `visualization` and plots $w(x)$ with each starting point joined to the minimum it reached, using NumPy and Matplotlib. It is optional: the table above lists the same start and end points, and nothing later uses this cell.

In [ ]:
# Optional: needs NumPy and Matplotlib, which JupyterLite and the course server provide.
try:
    import numpy as np
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("NumPy or Matplotlib is not available here; the table above shows the same results.")

if plt is not None:
    grid = np.linspace(-2.1, 2.1, 400)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(grid, grid ** 4 - 3 * grid ** 2 + grid, label="w(x) = x^4 - 3x^2 + x")
    for start, end in endpoints.items():
        ax.annotate("", xy=(end, double_well(end)), xytext=(start, double_well(start)),
                    arrowprops={"arrowstyle": "->", "alpha": 0.6})
        ax.plot([start], [double_well(start)], marker="o", markersize=4, color="grey")
    ax.set_xlabel("x")
    ax.set_ylabel("w(x)")
    ax.set_title("Each start rolls into the valley on its own side of the hump")
    ax.legend()
    plt.show()

### Step 7 — A first look at convexity

Convexity explains when the valley problem of Step 6 cannot happen. `find_chord_violation` tests the definition directly: it draws random pairs $a, b$ and weights $\lambda$ from a seeded generator and looks for a chord that passes *below* the graph. Finding one proves that a function is not convex; failing to find one is evidence, not proof, that it is convex. We also apply the second-derivative test: in one dimension, a twice-differentiable function is convex exactly when $L''(x) \ge 0$ everywhere.

In [ ]:
def find_chord_violation(f, low, high, trials=2000, seed=0):
    """Look for a, b and lam with f(lam*a + (1 - lam)*b) > lam*f(a) + (1 - lam)*f(b).

    Returns the first violating (a, b, lam) found, or None.
    """
    rng = random.Random(seed)
    for _ in range(trials):
        a, b, lam = rng.uniform(low, high), rng.uniform(low, high), rng.random()
        on_graph = f(lam * a + (1 - lam) * b)
        on_chord = lam * f(a) + (1 - lam) * f(b)
        if on_graph > on_chord + 1e-12:  # the margin ignores rounding error
            return a, b, lam
    return None


for name, f in (("quadratic", quadratic), ("abs", abs), ("double_well", double_well)):
    violation = find_chord_violation(f, -2.0, 2.0)
    if violation is None:
        print(f"{name:<12} no chord below the graph in 2000 trials (consistent with convexity)")
    else:
        a, b, lam = violation
        print(f"{name:<12} NOT convex: the chord from a = {a:.3f} to b = {b:.3f} passes below the graph")

print("\nSecond-derivative test for the double well, w''(x) = 12x^2 - 6:")
for x in (-1.0, -0.5, 0.0, 0.5, 1.0):
    print(f"  w''({x:+.1f}) = {double_well_curvature(x):+.1f}")

In [ ]:
# Tests for Step 7
assert find_chord_violation(quadratic, -2.0, 2.0) is None, "the quadratic is convex"
assert find_chord_violation(abs, -2.0, 2.0) is None, "|x| is convex, even though it has a kink at 0"
violation = find_chord_violation(double_well, -2.0, 2.0)
assert violation is not None, "the double well is not convex"
a, b, lam = violation
assert double_well(lam * a + (1 - lam) * b) > lam * double_well(a) + (1 - lam) * double_well(b), "the chord must really pass below the graph"
assert double_well_curvature(0.0) < 0 < double_well_curvature(1.5), "the curvature of the double well changes sign"
print("Step 7 tests passed.")

**Expected behaviour.** No violation is found for the quadratic or for $|x|$, which shows that convexity does not require a smooth function. For the double well a violating chord turns up almost at once. Its second derivative is negative between about $-0.71$ and $0.71$, where the graph bends downwards, so it is not convex; that hump is exactly what separates the two valleys of Step 6. The test cell prints `Step 7 tests passed.`

### Step 8 — Overflow and a stable $\log(1 + e^{x})$

A Python `float` cannot exceed about $1.8 \times 10^{308}$. Operations react differently when a result would be larger: multiplication and addition return `inf`, while `**` and `math.exp` raise `OverflowError`. Once an `inf` appears, expressions such as `inf - inf` produce `nan` ("not a number"), and `nan` spreads through every later calculation. `math.isfinite(v)` is `False` for both `inf` and `nan`.

The function $\operatorname{softplus}(x) = \log(1 + e^{x})$ appears in the loss of logistic regression and as a smooth version of $\max(0, x)$. Its direct formula fails at both ends:

- for large $x$, `math.exp(x)` overflows, although the answer is close to $x$;
- for very negative $x$, `1 + math.exp(x)` rounds to exactly `1.0`, so the result is `0.0` instead of the tiny positive value $\approx e^{x}$.

The rearrangement

$$\log(1 + e^{x}) = \max(x, 0) + \log\bigl(1 + e^{-|x|}\bigr)$$

never exponentiates a positive number, and `math.log1p(u)`, which computes $\log(1 + u)$ accurately for small $u$, keeps the tiny values.

In [ ]:
def softplus_naive(x):
    return math.log(1.0 + math.exp(x))


def softplus(x):
    """Numerically stable log(1 + exp(x))."""
    return max(x, 0.0) + math.log1p(math.exp(-abs(x)))


big = 1e200
print(f"1e200 * 1e200    -> {big * big}")
print(f"inf - inf        -> {math.inf - math.inf}")
for label, compute in (("1e200 ** 2", lambda: big ** 2), ("math.exp(1000)", lambda: math.exp(1000))):
    try:
        print(f"{label:<16} -> {compute()}")
    except OverflowError as error:
        print(f"{label:<16} -> OverflowError: {error}")

print(f"\n{'x':>7}  {'naive':>24}  {'stable':>24}")
for x in (-40.0, -1.0, 0.0, 1.0, 40.0, 800.0):
    try:
        naive = f"{softplus_naive(x):.17g}"
    except OverflowError:
        naive = "OverflowError"
    print(f"{x:>7.1f}  {naive:>24}  {softplus(x):>24.17g}")

In [ ]:
# Tests for Step 8
for x in (-5.0, -1.0, -0.5, 0.0, 0.5, 1.0, 5.0, 30.0):
    assert math.isclose(softplus(x), softplus_naive(x), rel_tol=1e-12), f"the two formulas should agree at x = {x}"
assert softplus(800.0) == 800.0, "for large x, softplus(x) equals x to double precision"
assert math.isclose(softplus(-40.0), math.exp(-40.0), rel_tol=1e-12), "for very negative x, softplus(x) is about exp(x)"
assert softplus_naive(-40.0) == 0.0, "the naive formula loses the tiny value entirely"
try:
    softplus_naive(800.0)
except OverflowError:
    pass
else:
    raise AssertionError("the naive formula should overflow at x = 800")
assert not math.isfinite(big * big) and not math.isfinite(math.inf - math.inf)
print("Step 8 tests passed.")

**Expected behaviour.** The first lines show `inf` from multiplication, `nan` from `inf - inf`, and `OverflowError` from `**` and from `math.exp`. In the table both formulas agree for moderate $x$, for example $0.69314718055994529 = \log 2$ at $x = 0$. At $x = -40$ the naive formula returns `0`, while the stable one returns about $4.248 \times 10^{-18}$, which is $e^{-40}$ to full precision. At $x = 800$ the naive formula raises `OverflowError`, while the stable one returns `800`. Deep-learning libraries compute cross-entropy from raw scores (logits) with rearrangements like this one. The test cell prints `Step 8 tests passed.`

### Step 9 — Bridge to machine learning: fit a line by gradient descent

Here is a tiny, fixed dataset: six inputs $x_i$ and targets $y_i$ that lie close to a straight line. A linear model predicts $\hat y_i = w x_i + b$, and we choose $w$ and $b$ to minimise the mean squared error

$$L(w, b) = \frac{1}{n} \sum_{i=1}^{n} (w x_i + b - y_i)^2 .$$

Writing $r_i = w x_i + b - y_i$ for the residuals, the chain rule gives

$$\frac{\partial L}{\partial w} = \frac{2}{n} \sum_{i=1}^{n} r_i\,x_i, \qquad \frac{\partial L}{\partial b} = \frac{2}{n} \sum_{i=1}^{n} r_i .$$

This loss is convex, so it has a single minimum, and for a straight line that minimum has a closed form, the **least-squares** solution:

$$w^\star = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sum_i (x_i - \bar x)^2}, \qquad b^\star = \bar y - w^\star \bar x,$$

where $\bar x$ and $\bar y$ are the means of the inputs and targets. We first check the gradient numerically, then fit the line by gradient descent, and finally compare the result with the formulas and with `statistics.linear_regression` from the standard library. The learning rate $0.05$ is below the stability limit of this loss, which is about $0.0626$.

In [ ]:
XS = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
YS = [3.1, 4.9, 7.2, 8.8, 11.1, 12.9]


def line_loss(theta):
    w, b = theta
    return mean_squared_error([w * x + b for x in XS], YS)


def line_grad(theta):
    w, b = theta
    residuals = [w * x + b - y for x, y in zip(XS, YS)]
    n = len(XS)
    return [2.0 / n * sum(r * x for r, x in zip(residuals, XS)), 2.0 / n * sum(residuals)]


def least_squares_line(xs, ys):
    """Closed-form slope and intercept that minimise the mean squared error."""
    n = len(xs)
    x_mean = sum(xs) / n
    y_mean = sum(ys) / n
    sxy = sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys))
    sxx = sum((x - x_mean) * (x - x_mean) for x in xs)
    slope = sxy / sxx
    return slope, y_mean - slope * x_mean


# 1. Check the analytic gradient before trusting it.
for theta in ([0.0, 0.0], [1.5, -0.3]):
    analytic = line_grad(theta)
    numerical = numerical_gradient(line_loss, theta)
    print(f"gradient at {theta}: analytic [{analytic[0]:.6f}, {analytic[1]:.6f}], "
          f"numerical [{numerical[0]:.6f}, {numerical[1]:.6f}]")

# 2. Fit the line by gradient descent from w = b = 0.
fit_thetas, fit_losses = gradient_descent(line_loss, line_grad, [0.0, 0.0], lr=0.05, steps=3000)
w_gd, b_gd = fit_thetas[-1]

# 3. Compare with the closed form and with the standard library.
w_ls, b_ls = least_squares_line(XS, YS)
w_stats, b_stats = statistics.linear_regression(XS, YS)
print(f"\n{'method':<28}  {'slope w':>10}  {'intercept b':>11}  {'MSE':>8}")
for name, w, b in (("gradient descent", w_gd, b_gd),
                   ("closed-form formulas", w_ls, b_ls),
                   ("statistics.linear_regression", w_stats, b_stats)):
    print(f"{name:<28}  {w:>10.6f}  {b:>11.6f}  {line_loss([w, b]):>8.5f}")
print("\nloss after 0, 10, 100, 1000 and 3000 steps:",
      ", ".join(f"{fit_losses[t]:.5f}" for t in (0, 10, 100, 1000, 3000)))

In [ ]:
# Tests for Step 9
for theta in ([0.0, 0.0], [1.5, -0.3], [3.0, 2.0]):
    for analytic, numerical in zip(line_grad(theta), numerical_gradient(line_loss, theta)):
        assert math.isclose(analytic, numerical, rel_tol=1e-7, abs_tol=1e-8), f"line_grad disagrees with a central difference at {theta}"
assert math.isclose(w_ls, 34.6 / 17.5) and math.isclose(b_ls, 1.08), "closed form: w = 34.6 / 17.5 and b = 8 - 3.5w = 1.08"
assert math.isclose(w_stats, w_ls) and math.isclose(b_stats, b_ls), "statistics.linear_regression should agree with the formulas"
assert math.isclose(w_gd, w_ls, rel_tol=1e-9) and math.isclose(b_gd, b_ls, rel_tol=1e-9), "gradient descent should reach the least-squares line"
assert vector_norm(line_grad([w_ls, b_ls])) < 1e-12, "the gradient vanishes at the least-squares solution"
for dw, db in ((0.01, 0.0), (0.0, 0.01), (-0.01, 0.01)):
    assert line_loss([w_ls + dw, b_ls + db]) > line_loss([w_ls, b_ls]), "moving away from the solution should increase the loss"
print("Step 9 tests passed.")

**Expected behaviour.** The analytic and numerical gradients agree. All three methods give the same line, $w \approx 1.977143$ and $b \approx 1.080000$, with a mean squared error of about $0.0185$. By hand: $\bar x = 3.5$, $\bar y = 8$, $\sum_i (x_i - \bar x)(y_i - \bar y) = 34.6$ and $\sum_i (x_i - \bar x)^2 = 17.5$, so $w^\star = 34.6/17.5$ and $b^\star = 8 - 3.5\,w^\star = 1.08$. The loss falls quickly at first and then slowly, because the loss is much steeper in one direction of $(w, b)$ than in another, just like the bowl of Step 3. The test cell prints `Step 9 tests passed.`

**Optional comparison.** The next cell is tagged `comparison`. It fits the same line with `numpy.polyfit`, a library least-squares routine, and confirms that it agrees with our closed-form formulas. It is optional, and nothing later depends on it.

In [ ]:
# Optional: needs NumPy, which JupyterLite and the course server provide.
try:
    import numpy as np
except ImportError:
    np = None
    print("NumPy is not available here; the table above already compares three methods.")

if np is not None:
    slope_np, intercept_np = np.polyfit(np.array(XS), np.array(YS), deg=1)
    agrees = math.isclose(slope_np, w_ls, rel_tol=1e-9) and math.isclose(intercept_np, b_ls, rel_tol=1e-9)
    print(f"numpy.polyfit: slope {slope_np:.6f}, intercept {intercept_np:.6f}")
    print(f"agrees with the closed-form formulas: {agrees}")

**Optional visualisation.** The next cell is tagged `visualization` and draws the data with the fitted line, next to the training loss over the first 300 steps, using Matplotlib. It is optional: the tables above contain the same results, and nothing later uses this cell.

In [ ]:
# Optional: needs Matplotlib, which JupyterLite and the course server provide.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Matplotlib is not available here; the tables above show the same results.")

if plt is not None:
    fig, (left_ax, right_ax) = plt.subplots(1, 2, figsize=(10, 4))
    left_ax.scatter(XS, YS, label="data")
    left_ax.plot([0.5, 6.5], [w_gd * 0.5 + b_gd, w_gd * 6.5 + b_gd], label=f"fit: y = {w_gd:.3f}x + {b_gd:.3f}")
    left_ax.set_xlabel("x")
    left_ax.set_ylabel("y")
    left_ax.set_title("Least-squares line found by gradient descent")
    left_ax.legend()
    right_ax.semilogy(fit_losses[:301])
    right_ax.set_xlabel("step")
    right_ax.set_ylabel("mean squared error (log scale)")
    right_ax.set_title("Training loss")
    plt.show()

## Exercises

Each exercise gives you a function stub. Replace the line `raise NotImplementedError(...)` with your code, run the cell, then run the `check(...)` cell below it. Until you do, the check prints, for example, `Exercise 1 (gradient_descent_with_stopping): not implemented yet`, and the rest of the notebook keeps working. You may call any function defined in the steps above.

### Exercise 1 — Gradient descent that knows when to stop

Write `gradient_descent_with_stopping(f, grad, theta0, lr, grad_tol=1e-6, loss_tol=1e-14, max_steps=10_000)`. Start from a copy of `theta0`, with `step = 0`, `previous_loss = None` and `loss = f(theta)`, and repeat:

1. compute the gradient at the current parameters;
2. call `stopping_reason(step, previous_loss, loss, gradient, grad_tol, loss_tol, max_steps)` from Step 5, and if it returns a reason, return the tuple `(theta, step, reason)`;
3. otherwise make one update $\theta \leftarrow \theta - \eta\,\nabla L(\theta)$, set `previous_loss` to the old loss, compute the new loss and add one to `step`.

So `step` counts the updates made so far. The check runs the bowl to convergence, caps a slow run, catches a diverging one, and shows the false convergence of a tiny learning rate.

In [ ]:
def gradient_descent_with_stopping(f, grad, theta0, lr, grad_tol=1e-6, loss_tol=1e-14, max_steps=10_000):
    """Gradient descent that stops by itself. Returns (theta, steps, reason)."""
    # TODO: follow the three steps above, using stopping_reason from Step 5.
    raise NotImplementedError("Exercise 1: implement gradient_descent_with_stopping")

In [ ]:
STOPPING_CASES = [
    ("the bowl from (-2, 1) with lr = 0.05 stops because the gradient norm is small",
     (bowl, bowl_grad, [-2.0, 1.0], 0.05), lambda result: result[2] == GRADIENT_SMALL),
    ("... within 1e-6 of the minimum (1, -2), well before the cap",
     (bowl, bowl_grad, [-2.0, 1.0], 0.05),
     lambda result: vector_norm([result[0][0] - 1.0, result[0][1] + 2.0]) < 1e-6 and 0 < result[1] < 1000),
    ("a cap of 50 steps stops a slow run (lr = 0.001) after exactly 50 updates",
     (bowl, bowl_grad, [-2.0, 1.0], 0.001, 1e-6, 1e-14, 50), lambda result: result[1:] == (50, CAP_REACHED)),
    ("lr = 0.2 makes the bowl diverge, and the run says so",
     (bowl, bowl_grad, [-2.0, 1.0], 0.2), lambda result: result[2] == DIVERGED),
    ("lr = 1e-13 with loss_tol = 1e-9 stops on the loss-change rule after one update (false convergence)",
     (bowl, bowl_grad, [-2.0, 1.0], 1e-13, 1e-6, 1e-9), lambda result: result[1:] == (1, LOSS_STALLED)),
    ("starting at the minimum stops at once, after 0 updates",
     (bowl, bowl_grad, [1.0, -2.0], 0.05), lambda result: result[1:] == (0, GRADIENT_SMALL)),
]
exercise_1_passed = check("Exercise 1 (gradient_descent_with_stopping)", gradient_descent_with_stopping, STOPPING_CASES)

### Exercise 2 — Choose a learning rate automatically

Write `best_learning_rate(f, grad, theta0, rates, steps)`. For each learning rate in `rates`, run `gradient_descent(f, grad, theta0, lr, steps)` from Step 3 and look at the final loss. Ignore runs whose final loss is not finite, because they diverged. Return the learning rate with the lowest final loss, or `None` if every run diverged.

This is a miniature hyperparameter search. In a real project you would compare losses on *validation* data that the model was not trained on, not the training loss used here.

In [ ]:
def best_learning_rate(f, grad, theta0, rates, steps):
    """Return the rate in `rates` whose run ends with the lowest finite loss, or None."""
    # TODO: run gradient_descent for each rate, skip runs whose last loss is not finite
    # (use math.isfinite), and keep the rate with the smallest last loss.
    raise NotImplementedError("Exercise 2: implement best_learning_rate")

In [ ]:
LEARNING_RATE_CASES = [
    ("quadratic, 20 steps: lr = 0.5 lands on the minimum in one step",
     (quadratic_vec, quadratic_vec_grad, [0.0], [0.001, 0.01, 0.1, 0.5, 1.5], 20), 0.5),
    ("bowl, 50 steps: 0.12 is unstable (its loss grows), so the best rate is 0.09",
     (bowl, bowl_grad, [-2.0, 1.0], [0.01, 0.05, 0.09, 0.12], 50), 0.09),
    ("the order of the rates does not matter",
     (bowl, bowl_grad, [-2.0, 1.0], [0.12, 0.09, 0.05, 0.01], 50), 0.09),
    ("every rate diverges within 1000 steps: return None",
     (quadratic_vec, quadratic_vec_grad, [0.0], [1.5, 2.0], 1000), None),
]
exercise_2_passed = check("Exercise 2 (best_learning_rate)", best_learning_rate, LEARNING_RATE_CASES)

### Exercise 3 — Fit a line through the origin

Some models have no intercept: $\hat y_i = w x_i$. For this model the mean squared error and its derivative are

$$L(w) = \frac{1}{n} \sum_{i=1}^{n} (w x_i - y_i)^2, \qquad L'(w) = \frac{2}{n} \sum_{i=1}^{n} (w x_i - y_i)\,x_i ,$$

and setting $L'(w) = 0$ gives the closed form $w^\star = \sum_i x_i y_i \big/ \sum_i x_i^2$.

Implement `fit_through_origin(xs, ys, lr, steps)`. It starts from $w = 0$, makes `steps` gradient-descent updates on $L(w)$ and returns the final $w$ as a number. You may write the loop yourself or call `gradient_descent` from Step 3 with suitable loss and gradient functions. The check compares your answer with the closed form, and also checks a single update against a hand calculation.

In [ ]:
def fit_through_origin(xs, ys, lr, steps):
    """Fit y = w * x by gradient descent on the mean squared error, starting from w = 0."""
    # TODO: compute L'(w) from the formula above and apply w <- w - lr * L'(w), `steps` times.
    raise NotImplementedError("Exercise 3: implement fit_through_origin")

In [ ]:
def closed_form_through_origin(xs, ys):
    return sum(x * y for x, y in zip(xs, ys)) / sum(x * x for x in xs)


ORIGIN_CASES = [
    ("one update from w = 0 on the Step 9 data: w = 0.01 * (2/6) * 202.6", (XS, YS, 0.01, 1), 0.01 * 2 / 6 * 202.6),
    ("200 updates on the Step 9 data reach the closed form", (XS, YS, 0.01, 200), closed_form_through_origin(XS, YS)),
    ("points exactly on y = 2.5x", ([1.0, 2.0, 4.0], [2.5, 5.0, 10.0], 0.02, 300), 2.5),
    ("a single point (2, -3): the line through it has slope -1.5", ([2.0], [-3.0], 0.1, 200), -1.5),
    ("zero updates leave w at its starting value 0", (XS, YS, 0.01, 0), 0.0),
]
exercise_3_passed = check("Exercise 3 (fit_through_origin)", fit_through_origin, ORIGIN_CASES, rel_tol=1e-9, abs_tol=1e-12)

### Exercise 4 (stretch) — Momentum

Plain gradient descent zig-zags across steep valleys and crawls along shallow ones, as the bowl showed in Step 3. **Momentum** keeps a running velocity that accumulates past gradients, so that consistent directions speed up while oscillating ones partly cancel. Use exactly this update, starting from $v_0 = 0$ (a list of zeros):

$$v_{t+1} = \beta\,v_t + \nabla L(\theta_t), \qquad \theta_{t+1} = \theta_t - \eta\,v_{t+1} .$$

Implement `momentum_descent(grad, theta0, lr, beta, steps)`, which returns the final parameters as a list. With $\beta = 0$ it must reduce to plain gradient descent.

A hand calculation to test yourself: on the quadratic from $x_0 = 0$ with $\eta = 0.1$ and $\beta = 0.9$, the first gradient is $-6$, so $v_1 = -6$ and $x_1 = 0.6$. The next gradient is $-4.8$, so $v_2 = 0.9 \times (-6) - 4.8 = -10.2$ and $x_2 = 0.6 + 1.02 = 1.62$.

In [ ]:
def momentum_descent(grad, theta0, lr, beta, steps):
    """Gradient descent with momentum; returns the final parameters as a list."""
    # TODO: start with theta = list(theta0) and velocity = [0.0] * len(theta0). At each step,
    # update the velocity first and then the parameters, exactly as in the formulas above.
    raise NotImplementedError("Exercise 4: implement momentum_descent")

In [ ]:
plain_thetas, _ = gradient_descent(bowl, bowl_grad, [-2.0, 1.0], 0.05, 100)
MOMENTUM_CASES = [
    ("one step on the quadratic (lr = 0.1, beta = 0.9) reaches 0.6", (quadratic_vec_grad, [0.0], 0.1, 0.9, 1), [0.6]),
    ("two steps reach 1.62, because the velocity carries over", (quadratic_vec_grad, [0.0], 0.1, 0.9, 2), [1.62]),
    ("three steps reach 2.814", (quadratic_vec_grad, [0.0], 0.1, 0.9, 3), [2.814]),
    ("beta = 0 reproduces plain gradient descent on the bowl", (bowl_grad, [-2.0, 1.0], 0.05, 0.0, 100), plain_thetas[-1]),
    ("beta = 0.5 gets within 1e-9 of the bowl's minimum in 100 steps, where plain descent is still about 8e-5 away",
     (bowl_grad, [-2.0, 1.0], 0.05, 0.5, 100), lambda result: vector_norm([result[0] - 1.0, result[1] + 2.0]) < 1e-9),
]
exercise_4_passed = check("Exercise 4 (momentum_descent)", momentum_descent, MOMENTUM_CASES, rel_tol=1e-9, abs_tol=1e-12)

## Common errors and debugging guidance

- **The loss grows, or becomes `inf` or `nan`.** The learning rate is too large for the steepest direction of the loss. Halve it until the loss falls steadily, and keep a `math.isfinite` check in the loop so that a diverging run stops with a clear message.
- **`OverflowError` instead of `inf`.** `math.exp` and `**` raise an exception when a result is too large, while `*` returns `inf`. Rewrite the formula in a stable form, as with `softplus`, or catch `OverflowError` and treat it as divergence, as `gradient_descent` does.
- **The loss falls, but painfully slowly.** The learning rate is too small, or the loss is much steeper in one direction than in another, as in the bowl. Standardising the input features (subtracting their mean and dividing by their standard deviation) usually helps a great deal; momentum (Exercise 4) is another remedy.
- **The loss rises even with a tiny learning rate.** Check the sign of the update: it must be `theta - lr * gradient`. Then check the gradient itself against a numerical gradient, as in Lesson 4.
- **The run stops at once and claims convergence.** A loss change below the tolerance can mean that the steps are tiny rather than that you have reached a minimum. Look at the gradient norm before you believe it.
- **Different runs give different answers.** On a non-convex loss the starting point decides the valley. Fix the seed of every `random.Random` you use, and try several starting points.
- **Every entry of a saved history is the same.** You appended the parameter list and then changed it in place, so every entry refers to one list. Build a new list at each step, as `gradient_descent` does.
- **A test fails although the numbers look equal.** Compare floats with `math.isclose`, and give an `abs_tol` whenever the expected value may be zero.

## Summary

- Training is minimisation: choose a loss $L(\theta)$ and search for the parameters that make it small.
- Gradient descent repeats $\theta_{t+1} = \theta_t - \eta\,\nabla L(\theta_t)$, stepping against the gradient, the direction of steepest increase.
- The learning rate decides the outcome: too small crawls, a good value converges quickly, and too large overshoots and diverges. Curvature sets the limit ($\eta < 2/a$ along a direction with second derivative $a$), and `math.isfinite` detects a run that has blown up.
- Stop on a small gradient norm or a small change in loss, always with an iteration cap, and record which rule fired.
- Non-convex losses have several valleys, so the result depends on the starting point; on a convex loss every local minimum is a global minimum.
- Floats overflow at about $1.8 \times 10^{308}$. Prefer stable formulas such as $\log(1 + e^{x}) = \max(x, 0) + \log(1 + e^{-|x|})$.
- Gradient descent on the mean squared error of a straight line reaches the same answer as the closed-form least-squares formulas, as the convexity of that loss promises.

**Next:** the same loop, with gradients from the chain rule, trains every model in the later courses; only the loss and the number of parameters change.

## Solutions

Reference solutions for the exercises follow. They have different names from the stubs, so running these cells never overwrites your own work. Each one is verified with the same `check` call and the same test cases as its exercise, which also proves that the tests accept a correct answer. Attempt each exercise before you read its solution.

In [ ]:
# Exercise 1: check whether to stop before every update
def gradient_descent_with_stopping_solution(f, grad, theta0, lr, grad_tol=1e-6, loss_tol=1e-14, max_steps=10_000):
    theta = list(theta0)
    step, previous_loss, loss = 0, None, f(theta)
    while True:
        gradient = grad(theta)
        reason = stopping_reason(step, previous_loss, loss, gradient, grad_tol, loss_tol, max_steps)
        if reason is not None:
            return theta, step, reason
        theta = [t - lr * g for t, g in zip(theta, gradient)]
        previous_loss, loss = loss, f(theta)
        step += 1


assert check("Exercise 1 (reference solution)", gradient_descent_with_stopping_solution, STOPPING_CASES), \
    "the reference solution must pass its own checks"

In [ ]:
# Exercise 2: keep the rate with the lowest finite final loss
def best_learning_rate_solution(f, grad, theta0, rates, steps):
    best_rate, best_loss = None, math.inf
    for lr in rates:
        _, losses = gradient_descent(f, grad, theta0, lr, steps)
        if math.isfinite(losses[-1]) and losses[-1] < best_loss:
            best_rate, best_loss = lr, losses[-1]
    return best_rate


assert check("Exercise 2 (reference solution)", best_learning_rate_solution, LEARNING_RATE_CASES), \
    "the reference solution must pass its own checks"

In [ ]:
# Exercise 3: L'(w) = (2/n) * sum((w*x - y) * x)
def fit_through_origin_solution(xs, ys, lr, steps):
    n = len(xs)
    w = 0.0
    for _ in range(steps):
        gradient = 2.0 / n * sum((w * x - y) * x for x, y in zip(xs, ys))
        w = w - lr * gradient
    return w


assert check("Exercise 3 (reference solution)", fit_through_origin_solution, ORIGIN_CASES,
             rel_tol=1e-9, abs_tol=1e-12), "the reference solution must pass its own checks"

In [ ]:
# Exercise 4: update the velocity, then the parameters
def momentum_descent_solution(grad, theta0, lr, beta, steps):
    theta = list(theta0)
    velocity = [0.0] * len(theta)
    for _ in range(steps):
        gradient = grad(theta)
        velocity = [beta * v + g for v, g in zip(velocity, gradient)]
        theta = [t - lr * v for t, v in zip(theta, velocity)]
    return theta


assert check("Exercise 4 (reference solution)", momentum_descent_solution, MOMENTUM_CASES,
             rel_tol=1e-9, abs_tol=1e-12), "the reference solution must pass its own checks"